# Experiment: Hierarchical DAG Manifold Structure

This notebook tests whether nonlinear encoders develop **context-dependent manifold structure**
when trained on distributions with **compositional/hierarchical structure**.

**Key hypothesis**: Linearity should struggle with compositional structure where a feature's
meaning changes depending on context. Consider a hierarchy:

- Feature A = "animal"
- Feature B = "has stripes"
- Feature C = "has wings"

When A+B co-activate → encode "zebra". When A+C co-activate → encode "bird".
Feature A's optimal encoding direction arguably should depend on which children are active.

**Implementation**: Use `DistributionStack` with multiple `DAGRandomWalkToRoot` distributions
to create separate hierarchical "concept clusters" (animals, vehicles, buildings, etc.)
that share similar structure but different features.

**Key questions**:
1. Do parent features (roots) show higher angular variance than leaf features?
2. Does the MLP outperform linear models more on hierarchical data than independent sparse data?
3. Is context-dependence (which children are active) reflected in Jacobian directions?

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from torch import Generator

from occhio import ToyModel, MLPAutoencoder
from occhio.autoencoder import TiedLinear, TiedLinearRelu
from occhio.distributions.sparse import SparseUniform
from occhio.distributions.dag import DAGRandomWalkToRoot
from occhio.distributions.base import DistributionStack
from occhio.analysis import (
    compute_feature_jacobians,
    angular_variance,
    jacobian_pca,
    direction_vs_context,
)

## 1. Configuration

Create multiple hierarchical DAG clusters using `DistributionStack`.
Each cluster represents a "concept domain" (e.g., animals, vehicles).

We use explicit adjacency matrices to ensure interpretable parent-child relationships.

In [ ]:
# Configuration
N_CLUSTERS = 10  # Number of hierarchical concept clusters
FEATURES_PER_CLUSTER = 20  # Features per cluster
N_FEATURES = N_CLUSTERS * FEATURES_PER_CLUSTER  # Total features
N_HIDDEN = 20  # 10:1 compression ratio
N_EPOCHS = 30000
BATCH_SIZE = 1000
N_JACOBIAN_SAMPLES = 1000

# DAG parameters
P_EDGE = 0.15  # Edge probability within each cluster
BETA = 0.8  # Value decay when walking up the DAG

# Importance decay
IMPORTANCE_DECAY = 0.996
IMPORTANCES = IMPORTANCE_DECAY ** torch.arange(N_FEATURES)

print(f"Configuration: {N_FEATURES} features -> {N_HIDDEN} hidden")
print(f"Compression ratio: {N_FEATURES / N_HIDDEN:.1f}:1")
print(f"Clusters: {N_CLUSTERS} x {FEATURES_PER_CLUSTER} features each")
print(f"DAG p_edge: {P_EDGE}, beta: {BETA}")

## 2. Build Hierarchical Distribution

Create a `DistributionStack` of `DAGRandomWalkToRoot` distributions.
Using `sampling_mode="single"` ensures only one concept cluster is active per sample,
forcing the model to learn distinct representations for each hierarchy.

In [ ]:
def create_hierarchical_distribution(seed=42, sampling_mode="single"):
    """Create a DistributionStack of DAG hierarchies.

    Args:
        seed: Random seed for reproducibility
        sampling_mode: "single" = one cluster per sample, "independent" = all clusters
    """
    clusters = []
    for i in range(N_CLUSTERS):
        cluster_gen = Generator().manual_seed(seed + i)
        dag = DAGRandomWalkToRoot(
            n_features=FEATURES_PER_CLUSTER,
            p_edge=P_EDGE,
            beta=BETA,
            generator=cluster_gen,
        )
        clusters.append(dag)

    stack_gen = Generator().manual_seed(seed + 1000)
    return DistributionStack(
        distributions=clusters,
        sampling_mode=sampling_mode,
        generator=stack_gen,
    )


def create_flat_sparse_distribution(seed=42):
    """Create a flat SparseUniform distribution for comparison.

    Matches the expected sparsity of the DAG distribution.
    """
    # Estimate sparsity from DAG distribution
    dag_dist = create_hierarchical_distribution(seed)
    dag_samples = dag_dist.sample(5000)
    estimated_density = (dag_samples > 0).float().mean().item()

    return SparseUniform(
        n_features=N_FEATURES,
        p_active=estimated_density,
        generator=Generator().manual_seed(seed),
    ), estimated_density


# Create and verify distributions
dag_dist = create_hierarchical_distribution()
flat_dist, estimated_density = create_flat_sparse_distribution()

# Sample and analyze
dag_samples = dag_dist.sample(5000)
flat_samples = flat_dist.sample(5000)

print(f"DAG distribution:")
print(f"  Sample shape: {dag_samples.shape}")
print(
    f"  Mean active features per sample: {(dag_samples > 0).sum(dim=1).float().mean():.1f}"
)
print(f"  Overall density: {(dag_samples > 0).float().mean():.4f}")

print(f"\nFlat sparse distribution:")
print(f"  Sample shape: {flat_samples.shape}")
print(
    f"  Mean active features per sample: {(flat_samples > 0).sum(dim=1).float().mean():.1f}"
)
print(f"  Target density: {estimated_density:.4f}")

In [ ]:
# Analyze DAG structure to identify roots, intermediates, and leaves
dag_dist = create_hierarchical_distribution(seed=42)

# Collect parent counts for each feature across all clusters
parent_counts_all = []
child_counts_all = []
cluster_ids = []

for cluster_idx, cluster_dag in enumerate(dag_dist.distributions):
    adj = cluster_dag.adjacency
    # Parent count = number of nodes pointing TO this node (column sum)
    parent_counts = adj.sum(dim=0).numpy()
    # Child count = number of nodes this node points TO (row sum)
    child_counts = adj.sum(dim=1).numpy()

    parent_counts_all.extend(parent_counts)
    child_counts_all.extend(child_counts)
    cluster_ids.extend([cluster_idx] * FEATURES_PER_CLUSTER)

parent_counts_all = np.array(parent_counts_all)
child_counts_all = np.array(child_counts_all)

# Classify features
is_root = parent_counts_all == 0
is_leaf = child_counts_all == 0
is_intermediate = ~is_root & ~is_leaf

print(f"Feature classification:")
print(f"  Roots (no parents): {is_root.sum()} features")
print(f"  Leaves (no children): {is_leaf.sum()} features")
print(f"  Intermediate: {is_intermediate.sum()} features")
print(f"\nMean parent count: {parent_counts_all.mean():.2f}")
print(f"Mean child count: {child_counts_all.mean():.2f}")

In [ ]:
# Visualize one cluster's DAG structure
cluster_idx = 0
cluster_dag = dag_dist.distributions[cluster_idx]

print(f"Cluster {cluster_idx} DAG structure:")
print(f"Adjacency matrix (rows=parents, cols=children):")
adj = cluster_dag.adjacency.numpy()

# Print edges
print("\nEdges:")
for i in range(FEATURES_PER_CLUSTER):
    children = [str(j) for j in range(FEATURES_PER_CLUSTER) if adj[i, j]]
    if children:
        print(f"  {i} → {', '.join(children)}")

print("\nSources and sinks:")
cluster_dag.print_sources_and_sinks()

## 3. Train Models

Train three architectures on both hierarchical DAG and flat sparse distributions:
1. **TiedLinear**: Linear baseline
2. **TiedLinearRelu**: Piecewise linear (ReLU decoder)
3. **MLPAutoencoder**: Smooth nonlinearity (GELU encoder)

In [ ]:
# Train on hierarchical DAG distribution
print("=" * 60)
print("Training on HIERARCHICAL DAG distribution")
print("=" * 60)

print("\nTraining TiedLinear...")
dag_linear_model = ToyModel(
    distribution=create_hierarchical_distribution(),
    ae=TiedLinear(n_features=N_FEATURES, n_hidden=N_HIDDEN),
    importances=IMPORTANCES,
)
dag_linear_losses, _ = dag_linear_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {dag_linear_losses[-1]:.6f}")

print("\nTraining TiedLinearRelu...")
dag_relu_model = ToyModel(
    distribution=create_hierarchical_distribution(),
    ae=TiedLinearRelu(n_features=N_FEATURES, n_hidden=N_HIDDEN),
    importances=IMPORTANCES,
)
dag_relu_losses, _ = dag_relu_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {dag_relu_losses[-1]:.6f}")

print("\nTraining MLPAutoencoder...")
dag_mlp_model = ToyModel(
    distribution=create_hierarchical_distribution(),
    ae=MLPAutoencoder(
        n_features=N_FEATURES,
        n_hidden=N_HIDDEN,
        encoder_hidden_dim=N_HIDDEN * 2,
        activation="gelu",
        decoder_activation="relu",
    ),
    importances=IMPORTANCES,
)
dag_mlp_losses, _ = dag_mlp_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {dag_mlp_losses[-1]:.6f}")

In [ ]:
# Train on flat sparse distribution for comparison
print("\n" + "=" * 60)
print("Training on FLAT SPARSE distribution (baseline)")
print("=" * 60)

flat_dist, _ = create_flat_sparse_distribution()

print("\nTraining TiedLinear...")
flat_linear_model = ToyModel(
    distribution=flat_dist,
    ae=TiedLinear(n_features=N_FEATURES, n_hidden=N_HIDDEN),
    importances=IMPORTANCES,
)
flat_linear_losses, _ = flat_linear_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {flat_linear_losses[-1]:.6f}")

print("\nTraining MLPAutoencoder...")
flat_mlp_model = ToyModel(
    distribution=create_flat_sparse_distribution()[0],
    ae=MLPAutoencoder(
        n_features=N_FEATURES,
        n_hidden=N_HIDDEN,
        encoder_hidden_dim=N_HIDDEN * 2,
        activation="gelu",
        decoder_activation="relu",
    ),
    importances=IMPORTANCES,
)
flat_mlp_losses, _ = flat_mlp_model.fit(
    n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, track_losses=True
)
print(f"Final loss: {flat_mlp_losses[-1]:.6f}")

In [ ]:
# Plot training losses
fig = make_subplots(
    rows=1, cols=2, subplot_titles=["DAG Distribution", "Flat Sparse Distribution"]
)

fig.add_trace(
    go.Scatter(y=dag_linear_losses, name="TiedLinear", mode="lines"), row=1, col=1
)
fig.add_trace(
    go.Scatter(y=dag_relu_losses, name="TiedLinearRelu", mode="lines"), row=1, col=1
)
fig.add_trace(go.Scatter(y=dag_mlp_losses, name="MLP", mode="lines"), row=1, col=1)

fig.add_trace(
    go.Scatter(y=flat_linear_losses, name="TiedLinear", mode="lines", showlegend=False),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(y=flat_mlp_losses, name="MLP", mode="lines", showlegend=False),
    row=1,
    col=2,
)

fig.update_layout(
    title="Training Loss: DAG vs Flat Sparse Distribution",
    height=400,
)
fig.update_yaxes(type="log")
fig.show()

## 4. Angular Variance Analysis

**Key hypothesis**: Parent features (roots) should show higher angular variance
because their optimal encoding depends on which children are co-active.

A → B: When A+B are active, A encodes "parent with child B active"
A → C: When A+C are active, A encodes "parent with child C active"

The MLP can adapt A's direction based on context; linear cannot.

In [ ]:
# Generate test samples for Jacobian analysis
test_dag_dist = create_hierarchical_distribution(seed=999)
test_dag_samples = test_dag_dist.sample(N_JACOBIAN_SAMPLES * 2)

test_flat_dist, _ = create_flat_sparse_distribution(seed=999)
test_flat_samples = test_flat_dist.sample(N_JACOBIAN_SAMPLES * 2)

print(f"DAG test samples: {test_dag_samples.shape}")
print(f"Flat test samples: {test_flat_samples.shape}")

In [ ]:
def compute_all_angular_variances(
    model, samples, n_samples_per_feature=500, verbose=True
):
    """Compute angular variance for all features."""
    n_features = model.n_features
    avs = []

    for feat_idx in range(n_features):
        # Filter for samples where this feature is active
        active_mask = samples[:, feat_idx] > 0
        active_samples = samples[active_mask]

        if len(active_samples) < 10:
            avs.append(np.nan)
            continue

        active_samples = active_samples[:n_samples_per_feature]

        jacs = compute_feature_jacobians(model, feat_idx, active_samples)
        av = angular_variance(jacs)
        avs.append(av)

        if verbose and feat_idx % 50 == 0:
            print(f"  Feature {feat_idx}: AV = {av:.6f} (n={len(active_samples)})")

    return np.array(avs)


print("Computing angular variance for DAG MLP...")
dag_mlp_avs = compute_all_angular_variances(dag_mlp_model, test_dag_samples)

print("\nComputing angular variance for DAG Linear...")
dag_linear_avs = compute_all_angular_variances(dag_linear_model, test_dag_samples)

print("\nComputing angular variance for Flat MLP...")
flat_mlp_avs = compute_all_angular_variances(flat_mlp_model, test_flat_samples)

In [ ]:
# Compare angular variance by feature role (root/intermediate/leaf)
print("=" * 60)
print("ANGULAR VARIANCE BY FEATURE ROLE (DAG MLP)")
print("=" * 60)

valid_mask = ~np.isnan(dag_mlp_avs)

root_avs = dag_mlp_avs[is_root & valid_mask]
leaf_avs = dag_mlp_avs[is_leaf & valid_mask]
intermediate_avs = dag_mlp_avs[is_intermediate & valid_mask]

print(f"\nRoots (no parents):")
print(
    f"  N = {len(root_avs)}, Mean AV = {root_avs.mean():.6f}, Std = {root_avs.std():.6f}"
)

print(f"\nIntermediates:")
print(
    f"  N = {len(intermediate_avs)}, Mean AV = {intermediate_avs.mean():.6f}, Std = {intermediate_avs.std():.6f}"
)

print(f"\nLeaves (no children):")
print(
    f"  N = {len(leaf_avs)}, Mean AV = {leaf_avs.mean():.6f}, Std = {leaf_avs.std():.6f}"
)

# Statistical comparison
from scipy import stats

if len(root_avs) > 5 and len(leaf_avs) > 5:
    t_stat, p_val = stats.ttest_ind(root_avs, leaf_avs)
    print(f"\nt-test (Roots vs Leaves): t = {t_stat:.3f}, p = {p_val:.4e}")

    if root_avs.mean() > leaf_avs.mean() and p_val < 0.05:
        print("=> CONFIRMED: Root features have significantly higher angular variance!")
        print(
            "   Parent encoding is context-dependent (depends on which children are active)"
        )
    elif leaf_avs.mean() > root_avs.mean() and p_val < 0.05:
        print("=> UNEXPECTED: Leaf features have higher angular variance than roots")
    else:
        print("=> No significant difference between roots and leaves")

In [ ]:
# Visualize angular variance by feature role
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "AV by Feature Index (DAG MLP)",
        "AV Distribution by Role",
        "AV vs Parent Count",
        "AV vs Child Count",
    ],
)

# AV by feature index, colored by role
colors = [
    "red" if is_root[i] else ("green" if is_leaf[i] else "blue")
    for i in range(N_FEATURES)
]
fig.add_trace(
    go.Scatter(
        x=list(range(N_FEATURES)),
        y=dag_mlp_avs,
        mode="markers",
        marker=dict(color=colors, size=5),
        name="Features",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Box plots by role
fig.add_trace(go.Box(y=root_avs, name="Roots", marker_color="red"), row=1, col=2)
fig.add_trace(
    go.Box(y=intermediate_avs, name="Intermediate", marker_color="blue"), row=1, col=2
)
fig.add_trace(go.Box(y=leaf_avs, name="Leaves", marker_color="green"), row=1, col=2)

# AV vs parent count
fig.add_trace(
    go.Scatter(
        x=parent_counts_all[valid_mask],
        y=dag_mlp_avs[valid_mask],
        mode="markers",
        marker=dict(size=5, opacity=0.6),
        name="AV vs Parents",
        showlegend=False,
    ),
    row=2,
    col=1,
)

# AV vs child count
fig.add_trace(
    go.Scatter(
        x=child_counts_all[valid_mask],
        y=dag_mlp_avs[valid_mask],
        mode="markers",
        marker=dict(size=5, opacity=0.6),
        name="AV vs Children",
        showlegend=False,
    ),
    row=2,
    col=2,
)

fig.update_xaxes(title_text="Feature Index", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=1)
fig.update_yaxes(title_text="Angular Variance", row=1, col=2)
fig.update_xaxes(title_text="Parent Count", row=2, col=1)
fig.update_yaxes(title_text="Angular Variance", row=2, col=1)
fig.update_xaxes(title_text="Child Count", row=2, col=2)
fig.update_yaxes(title_text="Angular Variance", row=2, col=2)

fig.update_layout(height=700, title_text="Angular Variance Analysis by DAG Role")
fig.show()

In [ ]:
# Compare DAG vs Flat distributions
print("=" * 60)
print("COMPARISON: DAG vs FLAT SPARSE DISTRIBUTIONS")
print("=" * 60)

dag_valid = dag_mlp_avs[~np.isnan(dag_mlp_avs)]
flat_valid = flat_mlp_avs[~np.isnan(flat_mlp_avs)]

print(f"\nDAG MLP:")
print(f"  Mean AV: {dag_valid.mean():.6f}")
print(f"  Max AV: {dag_valid.max():.6f}")
print(f"  Std AV: {dag_valid.std():.6f}")

print(f"\nFlat MLP:")
print(f"  Mean AV: {flat_valid.mean():.6f}")
print(f"  Max AV: {flat_valid.max():.6f}")
print(f"  Std AV: {flat_valid.std():.6f}")

t_stat, p_val = stats.ttest_ind(dag_valid, flat_valid)
print(f"\nt-test (DAG vs Flat): t = {t_stat:.3f}, p = {p_val:.4e}")

if dag_valid.mean() > flat_valid.mean() and p_val < 0.05:
    print("=> CONFIRMED: Hierarchical structure promotes more manifold bending!")
elif flat_valid.mean() > dag_valid.mean() and p_val < 0.05:
    print("=> Flat distribution shows more manifold structure (unexpected)")
else:
    print("=> No significant difference in manifold structure")

## 5. Context Dependence Analysis

For high-AV parent features, analyze which children most influence the encoding direction.
If the "animal" feature's encoding direction changes based on whether "stripes" or "wings"
is active, we should see strong correlations in the direction_vs_context analysis.

In [ ]:
# Find high-AV parent features
root_indices = np.where(is_root)[0]
root_avs_with_idx = [
    (idx, dag_mlp_avs[idx]) for idx in root_indices if not np.isnan(dag_mlp_avs[idx])
]
root_avs_with_idx.sort(key=lambda x: x[1], reverse=True)

print("Top 10 highest-AV root features:")
for idx, av in root_avs_with_idx[:10]:
    cluster = idx // FEATURES_PER_CLUSTER
    local_idx = idx % FEATURES_PER_CLUSTER
    n_children = child_counts_all[idx]
    print(
        f"  Feature {idx} (cluster {cluster}, local {local_idx}): AV = {av:.4f}, children = {n_children}"
    )

In [ ]:
# Analyze direction vs context for a high-AV root feature
if root_avs_with_idx:
    top_root_idx = root_avs_with_idx[0][0]
    top_root_av = root_avs_with_idx[0][1]
    cluster = top_root_idx // FEATURES_PER_CLUSTER
    local_idx = top_root_idx % FEATURES_PER_CLUSTER

    print(f"Analyzing feature {top_root_idx} (cluster {cluster}, local {local_idx})")
    print(f"Angular variance: {top_root_av:.6f}")

    # Get its children (within the same cluster)
    cluster_dag = dag_dist.distributions[cluster]
    adj = cluster_dag.adjacency.numpy()
    children_local = np.where(adj[local_idx, :])[0]
    children_global = [cluster * FEATURES_PER_CLUSTER + c for c in children_local]

    print(f"Children (global indices): {children_global}")

    # Get active samples
    active_mask = test_dag_samples[:, top_root_idx] > 0
    active_samples = test_dag_samples[active_mask][:500]

    if len(active_samples) >= 10:
        jacs = compute_feature_jacobians(dag_mlp_model, top_root_idx, active_samples)
        context_result = direction_vs_context(
            dag_mlp_model,
            feature_idx=top_root_idx,
            inputs=active_samples,
            jacobians=jacs,
        )

        print(f"\nMost influential co-active features:")
        for feat_idx, magnitude in context_result["most_influential"][:10]:
            is_child = feat_idx in children_global
            marker = " [CHILD]" if is_child else ""
            print(
                f"  Feature {feat_idx}: correlation magnitude = {magnitude:.4f}{marker}"
            )

        # Check if children are among most influential
        top_influential = [f[0] for f in context_result["most_influential"][:10]]
        children_in_top = [c for c in children_global if c in top_influential]
        print(
            f"\nChildren among top 10 influential: {len(children_in_top)}/{len(children_global)}"
        )
    else:
        print("Not enough active samples for context analysis")

## 6. Reconstruction Quality Comparison

Compare MLP vs Linear improvement on hierarchical vs flat distributions.
If hierarchical structure benefits from nonlinearity, we expect larger MLP gains on DAG data.

In [ ]:
# Compute reconstruction loss on held-out data
eval_dag_dist = create_hierarchical_distribution(seed=12345)
eval_dag_samples = eval_dag_dist.sample(5000)

eval_flat_dist, _ = create_flat_sparse_distribution(seed=12345)
eval_flat_samples = eval_flat_dist.sample(5000)


def compute_reconstruction_loss(model, samples):
    with torch.no_grad():
        result = model.ae(samples)
        reconstructed = result[0] if isinstance(result, tuple) else result
        mse = ((samples - reconstructed) ** 2).mean().item()
    return mse


dag_linear_loss = compute_reconstruction_loss(dag_linear_model, eval_dag_samples)
dag_relu_loss = compute_reconstruction_loss(dag_relu_model, eval_dag_samples)
dag_mlp_loss = compute_reconstruction_loss(dag_mlp_model, eval_dag_samples)

flat_linear_loss = compute_reconstruction_loss(flat_linear_model, eval_flat_samples)
flat_mlp_loss = compute_reconstruction_loss(flat_mlp_model, eval_flat_samples)

print("=" * 60)
print("RECONSTRUCTION LOSS COMPARISON")
print("=" * 60)

print("\nHierarchical DAG distribution:")
print(f"  TiedLinear:      {dag_linear_loss:.6f}")
print(f"  TiedLinearRelu:  {dag_relu_loss:.6f}")
print(f"  MLPAutoencoder:  {dag_mlp_loss:.6f}")
dag_improvement = (dag_linear_loss - dag_mlp_loss) / dag_linear_loss * 100
print(f"  MLP improvement: {dag_improvement:.2f}%")

print("\nFlat Sparse distribution:")
print(f"  TiedLinear:      {flat_linear_loss:.6f}")
print(f"  MLPAutoencoder:  {flat_mlp_loss:.6f}")
flat_improvement = (flat_linear_loss - flat_mlp_loss) / flat_linear_loss * 100
print(f"  MLP improvement: {flat_improvement:.2f}%")

print("\n" + "=" * 60)
if dag_improvement > flat_improvement:
    print(
        f"=> DAG benefits MORE from nonlinearity (+{dag_improvement - flat_improvement:.2f}pp)"
    )
    print("   Hierarchical structure requires context-dependent encoding!")
else:
    print(
        f"=> Flat benefits MORE from nonlinearity (+{flat_improvement - dag_improvement:.2f}pp)"
    )

## 7. 3D Visualization (Low-Dimensional)

Train a small model with n_hidden=3 to directly visualize Jacobian directions on a sphere.
Color by which children are active to see context-dependent direction shifts.

In [ ]:
# Smaller models for 3D visualization
N_CLUSTERS_3D = 5
FEATURES_PER_CLUSTER_3D = 10
N_FEATURES_3D = N_CLUSTERS_3D * FEATURES_PER_CLUSTER_3D
N_HIDDEN_3D = 3
N_EPOCHS_3D = 3000


def create_hierarchical_distribution_3d(seed=42):
    clusters = []
    for i in range(N_CLUSTERS_3D):
        cluster_gen = Generator().manual_seed(seed + i)
        dag = DAGRandomWalkToRoot(
            n_features=FEATURES_PER_CLUSTER_3D,
            p_edge=0.2,
            beta=0.8,
            generator=cluster_gen,
        )
        clusters.append(dag)

    stack_gen = Generator().manual_seed(seed + 1000)
    return DistributionStack(
        distributions=clusters,
        sampling_mode="single",
        generator=stack_gen,
    )


print("Training 3D MLP on hierarchical DAG...")
mlp_3d = ToyModel(
    distribution=create_hierarchical_distribution_3d(),
    ae=MLPAutoencoder(
        n_features=N_FEATURES_3D,
        n_hidden=N_HIDDEN_3D,
        encoder_hidden_dim=N_HIDDEN_3D * 2,
        activation="gelu",
        decoder_activation="relu",
    ),
)
mlp_3d.fit(n_epochs=N_EPOCHS_3D, batch_size=BATCH_SIZE)
print("Done.")

In [ ]:
# Analyze 3D model structure
dag_3d = create_hierarchical_distribution_3d(seed=42)
test_samples_3d = dag_3d.sample(2000)

# Find a root feature with children
cluster_idx = 0
cluster_dag = dag_3d.distributions[cluster_idx]
adj = cluster_dag.adjacency.numpy()

# Find roots (no parents) with children
for local_idx in range(FEATURES_PER_CLUSTER_3D):
    has_parents = adj[:, local_idx].any()
    has_children = adj[local_idx, :].any()
    if not has_parents and has_children:
        global_idx = cluster_idx * FEATURES_PER_CLUSTER_3D + local_idx
        children_local = np.where(adj[local_idx, :])[0]
        children_global = [
            cluster_idx * FEATURES_PER_CLUSTER_3D + c for c in children_local
        ]
        print(
            f"Root feature {global_idx} (local {local_idx}) has children: {children_global}"
        )

        # Check if enough active samples
        active_mask = test_samples_3d[:, global_idx] > 0
        n_active = active_mask.sum().item()
        print(f"  Active in {n_active} samples")

        if n_active >= 50:
            target_feature = global_idx
            target_children = children_global
            break

In [ ]:
# 3D visualization of Jacobian directions
if "target_feature" in dir():
    active_mask = test_samples_3d[:, target_feature] > 0
    active_samples_3d = test_samples_3d[active_mask][:500]

    jacs_3d = compute_feature_jacobians(mlp_3d, target_feature, active_samples_3d)
    jacs_normed = jacs_3d / jacs_3d.norm(dim=1, keepdim=True).clamp(min=1e-8)

    # Color by which child is most active
    if len(target_children) >= 2:
        child1, child2 = target_children[:2]
        child1_values = active_samples_3d[:, child1].numpy()
        child2_values = active_samples_3d[:, child2].numpy()
        # Color: ratio of child1 to sum of both children
        colors = child1_values / (child1_values + child2_values + 1e-8)
        colorbar_title = f"Child {child1} / (Child {child1} + {child2})"
    else:
        colors = active_samples_3d[:, target_children[0]].numpy()
        colorbar_title = f"Child {target_children[0]} value"

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=jacs_normed[:, 0].detach().cpu().numpy(),
                y=jacs_normed[:, 1].detach().cpu().numpy(),
                z=jacs_normed[:, 2].detach().cpu().numpy(),
                mode="markers",
                marker=dict(
                    size=4,
                    color=colors,
                    colorscale="RdBu",
                    colorbar=dict(title=colorbar_title),
                    opacity=0.8,
                ),
            )
        ]
    )

    fig.update_layout(
        title=f"Jacobian Directions for Root Feature {target_feature}<br>Colored by Child Activity",
        scene=dict(
            xaxis_title="Hidden Dim 0",
            yaxis_title="Hidden Dim 1",
            zaxis_title="Hidden Dim 2",
        ),
        height=600,
    )
    fig.show()

    # Compute angular variance
    av_3d = angular_variance(jacs_3d)
    print(f"Angular variance for feature {target_feature}: {av_3d:.6f}")
else:
    print("No suitable root feature found with enough active samples")

## 8. Summary

In [ ]:
print("=" * 70)
print("EXPERIMENT SUMMARY: Hierarchical DAG Manifold Structure")
print("=" * 70)

print("\nCONFIGURATION:")
print(f"  Features: {N_FEATURES} ({N_CLUSTERS} clusters x {FEATURES_PER_CLUSTER})")
print(f"  Hidden: {N_HIDDEN} (compression {N_FEATURES / N_HIDDEN:.0f}:1)")
print(f"  DAG structure: p_edge={P_EDGE}, beta={BETA}")

print("\n1. ANGULAR VARIANCE BY FEATURE ROLE:")
if len(root_avs) > 0 and len(leaf_avs) > 0:
    print(f"   Roots (parents): {root_avs.mean():.6f} ± {root_avs.std():.6f}")
    print(f"   Leaves (children): {leaf_avs.mean():.6f} ± {leaf_avs.std():.6f}")
    if len(intermediate_avs) > 0:
        print(
            f"   Intermediate: {intermediate_avs.mean():.6f} ± {intermediate_avs.std():.6f}"
        )

print("\n2. HIERARCHICAL vs FLAT COMPARISON:")
print(f"   DAG MLP mean AV: {dag_valid.mean():.6f}")
print(f"   Flat MLP mean AV: {flat_valid.mean():.6f}")

print("\n3. RECONSTRUCTION IMPROVEMENT:")
print(f"   DAG: MLP vs Linear = {dag_improvement:.2f}%")
print(f"   Flat: MLP vs Linear = {flat_improvement:.2f}%")
print(f"   Difference: {dag_improvement - flat_improvement:+.2f}pp")

print("\n" + "=" * 70)
print("KEY FINDINGS:")
print("=" * 70)

if root_avs.mean() > leaf_avs.mean():
    print("\n✓ Parent features show HIGHER angular variance than leaf features")
    print("  => Encoding direction depends on which children are active")
    print("  => The 'animal' feature points differently for 'zebra' vs 'bird'")
else:
    print("\n✗ No clear parent > leaf angular variance pattern")

if dag_valid.mean() > flat_valid.mean():
    print("\n✓ Hierarchical structure produces MORE manifold bending than flat sparse")
    print("  => Compositional structure benefits from context-dependent encoding")
else:
    print("\n✗ Flat sparse shows similar or more manifold structure")

if dag_improvement > flat_improvement:
    print("\n✓ MLP gains are LARGER on hierarchical data")
    print("  => Nonlinearity is especially valuable for compositional structure")
else:
    print("\n✗ MLP gains are similar or larger on flat data")